In [ ]:
import os
import subprocess
import time
import numpy as np
import faiss
import ollama
import gradio as gr
from pypdf import PdfReader

print("1. Установка системных зависимостей и Ollama")

!apt-get update && apt-get install -y zstd lshw pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama gradio faiss-cpu pypdf numpy

print("2. Подготовка окружения GPU")
!pkill -9 ollama

os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia"
os.environ["PATH"] += ":/usr/local/cuda/bin"
os.environ["OLLAMA_INTEL_GPU"] = "0"

print("\n--- 3. Запуск сервера Ollama ---")
process = subprocess.Popen(['ollama', 'serve'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.PIPE)

time.sleep(15)

print("4. Загрузка моделей (это может занять время)")

!ollama pull bge-m3
!ollama pull qwen3:4b

print("5. Проверка состояния")
!nvidia-smi
print("Список загруженных моделей:")
!ollama list

print("Готово. Запускаем код RAG системы.")

PDF_DIR = "Database"
DB_DIR = "Vector_embeddings_database/3GPP_Database"
MODEL_NAME = "qwen3:4b"
EMBED_MODEL = "bge-m3"

client = ollama.Client(host='http://127.0.0.1:11434')

def load_db():
    index_path = os.path.join(DB_DIR, "index.faiss")
    texts_path = os.path.join(DB_DIR, "texts.npy")
    if os.path.exists(index_path) and os.path.exists(texts_path):
        print(f"База загружена успешно")
        index = faiss.read_index(index_path)
        chunks = np.load(texts_path, allow_pickle=True, mmap_mode='r')
        return index, chunks
    print("Ошибка: База не найдена по указанному пути")
    return None, None

index, all_chunks = load_db()

def get_context(query, index, chunks, k=3):
    resp = client.embeddings(model=EMBED_MODEL, prompt=query)
    query_emb = np.array([resp['embedding']]).astype('float32')
    _, indices = index.search(query_emb, k)
    context_parts = [str(chunks[i]) for i in indices[0] if i != -1 and i < len(chunks)]
    return "\n---\n".join(context_parts)

def chat(message, history):
    if not message or index is None:
        yield "Система не готова или запрос пуст."
        return

    try:
        context = get_context(message, index, all_chunks, k=4)

        raw_prompt = (
            f"<|im_start|>system\n"
            f"Ты — ведущий инженер по сетям 5G и эксперт по стандартам 3GPP. "
            f"Твоя цель: дать развернутый, понятный и технически грамотный ответ на русском языке. \n\n"
            f"ПРАВИЛА:\n"
            f"1. Используй предоставленную ДОКУМЕНТАЦИЮ как главный источник истины.\n"
            f"2. Пиши структурированно: используй заголовки, жирный шрифт для терминов и маркированные списки.\n"
            f"3. Если в тексте есть сложные аббревиатуры (например, AMF, gNB, PDU), кратко поясняй их значение.\n"
            f"4. Не используй теги <think>. Твой тон: профессиональный, дружелюбный, экспертный.\n"
            f"5. Если информации в документах недостаточно, ответь на основе того, что есть, но упомяни, что это согласно доступным фрагментам.\n"
            f"<|im_end|>\n"
            f"<|im_start|>user\n"
            f"КОНТЕКСТ ИЗ ДОКУМЕНТАЦИИ:\n{context}\n\n"
            f"ВОПРОС ПОЛЬЗОВАТЕЛЯ: {message}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

        stream = client.generate(
            model=MODEL_NAME,
            prompt=raw_prompt,
            stream=True,
            options={
                "num_ctx": 4096,
                "temperature": 0.3,
                "top_p": 0.9,
                "stop": ["<|im_start|>", "<|im_end|>", "system\n", "user\n"]
            }
        )

        full_response = ""
        for chunk in stream:
            content = chunk.get('response', '')
            full_response += content

            display_text = full_response.split("</think>")[-1].strip()
            yield display_text

    except Exception as e:
        yield f"Произошла техническая ошибка: {str(e)}"

demo = gr.ChatInterface(
    fn=chat,
    title="📡 5G Ассистент (RAG система)",
    description="Локальный доступ к базе 3GPP через языковую модель Qwen3:4B.",
)

if __name__ == "__main__":
    gr.close_all()
    demo.launch(
        share=False,
        server_name="127.0.0.1",
        server_port=7860,
        debug=True
    )


--- 2. Подготовка окружения GPU ---

--- 3. Запуск сервера Ollama ---

--- 4. Загрузка моделей (это может занять время) ---



--- 5. Проверка состояния ---
Fri Mar 27 17:51:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Closing server running on port: 7860

--- Проверь, что Ollama запущена! ---
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
